# Bayes' Theorem in Spam Detection: Naive Bayes on the SpamAssassin Corpus
This notebook goes with the report *Case Study on the Application of Bayes' Theorem in Spam Detection*. It reproduces every number in the report: the dataset counts, the worked "click" example, Table 1 and the threshold analysis.

Requirements: `pip install scikit-learn numpy`

## 1. Download the dataset
The SpamAssassin public corpus is downloaded into `../data` if it isn't there already.

In [1]:
import os, re, tarfile, urllib.request, email, email.policy, html
import numpy as np

DATA = '../data'
BASE = 'https://spamassassin.apache.org/old/publiccorpus/'
FILES = ['20030228_easy_ham', '20030228_easy_ham_2', '20030228_hard_ham', '20030228_spam', '20050311_spam_2']
os.makedirs(DATA, exist_ok=True)
for f in FILES:
    archive = os.path.join(DATA, f + '.tar.bz2')
    if not os.path.exists(archive):
        urllib.request.urlretrieve(BASE + f + '.tar.bz2', archive)
    with tarfile.open(archive) as t:
        if not os.path.isdir(os.path.join(DATA, t.getnames()[0].split('/')[0])):
            t.extractall(DATA)
print(sorted(d for d in os.listdir(DATA) if os.path.isdir(os.path.join(DATA, d))))

['easy_ham', 'easy_ham_2', 'hard_ham', 'spam', 'spam_2']


## 2. Parse and normalize the emails
Keep the subject and all text parts, strip HTML tags, lowercase the text, and replace URLs, email addresses and numbers with placeholder tokens.

In [2]:
def text_of(path):
    with open(path, 'rb') as f:
        msg = email.message_from_bytes(f.read(), policy=email.policy.default)
    parts = [str(msg['subject'] or '')]
    for p in msg.walk():
        if p.get_content_maintype() == 'text':
            try:
                t = p.get_content()
            except Exception:
                raw = p.get_payload(decode=True)
                t = raw.decode('latin-1', 'ignore') if raw else ''
            if p.get_content_subtype() == 'html':
                t = html.unescape(re.sub(r'<[^>]+>', ' ', t))
            parts.append(t)
    return '\n'.join(parts)

def normalize(t):
    t = t.lower()
    t = re.sub(r'(https?://|www\.)\S+', ' URLTOKEN ', t)
    t = re.sub(r'\S+@\S+', ' EMAILTOKEN ', t)
    t = re.sub(r'\d+(\.\d+)?', ' NUMTOKEN ', t)
    return t

X, y = [], []
for folder, label in [('easy_ham', 0), ('easy_ham_2', 0), ('hard_ham', 0), ('spam', 1), ('spam_2', 1)]:
    for fn in os.listdir(os.path.join(DATA, folder)):
        if fn == 'cmds':
            continue
        X.append(normalize(text_of(os.path.join(DATA, folder, fn))))
        y.append(label)
y = np.array(y)
print(f'Total: {len(y)}  spam: {y.sum()} ({y.mean():.1%})  ham: {(y == 0).sum()}')

Total: 6046  spam: 1896 (31.4%)  ham: 4150


## 3. Train/test split
An 80/20 stratified split, so both parts keep the original spam-to-ham ratio.

In [3]:
from sklearn.model_selection import train_test_split
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print('train', len(ytr), ' test', len(yte), ' test spam', yte.sum())

train 4836  test 1210  test spam 379


## 4. Worked example of Bayes' theorem
P(spam | \"click\") computed directly from training-set frequencies.

In [4]:
prior = ytr.mean()
has = np.array([bool(re.search(r'\bclick\b', t)) for t in Xtr])
p_w_spam, p_w_ham = has[ytr == 1].mean(), has[ytr == 0].mean()
posterior = p_w_spam * prior / (p_w_spam * prior + p_w_ham * (1 - prior))
print(f'P(spam) = {prior:.3f}, P(click|spam) = {p_w_spam:.3f}, P(click|ham) = {p_w_ham:.3f}')
print(f'P(spam|click) = {posterior:.2f}')

P(spam) = 0.314, P(click|spam) = 0.544, P(click|ham) = 0.047
P(spam|click) = 0.84


## 5. Tokenization, vectorization and training
Tokens are words of 2+ letters. English stop words are removed and words must appear in at least 2 messages. Three naive Bayes variants are compared.

In [5]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

TOK = r'(?u)\b[a-zA-Z][a-zA-Z]+\b'
models = {
    'Multinomial (counts)': lambda: make_pipeline(CountVectorizer(token_pattern=TOK, stop_words='english', min_df=2), MultinomialNB(alpha=1.0)),
    'Bernoulli (binary)':   lambda: make_pipeline(CountVectorizer(token_pattern=TOK, stop_words='english', min_df=2, binary=True), BernoulliNB(alpha=1.0)),
    'Multinomial (TF-IDF)': lambda: make_pipeline(TfidfVectorizer(token_pattern=TOK, stop_words='english', min_df=2), MultinomialNB(alpha=0.1)),
}
fitted = {}
cv = StratifiedKFold(5, shuffle=True, random_state=42)
for name, build in models.items():
    m = build().fit(Xtr, ytr); fitted[name] = m
    pred, prob = m.predict(Xte), m.predict_proba(Xte)[:, 1]
    tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
    cvs = cross_val_score(build(), X, y, cv=cv)
    print(f'{name:22s} acc={accuracy_score(yte, pred):.4f} prec={precision_score(yte, pred):.4f} '
          f'rec={recall_score(yte, pred):.4f} f1={f1_score(yte, pred):.4f} spec={tn / (tn + fp):.4f} '
          f'auc={roc_auc_score(yte, prob):.4f} cv={cvs.mean():.4f}±{cvs.std():.4f}  [TP={tp} FN={fn} FP={fp} TN={tn}]')
print('Vocabulary size:', len(fitted['Multinomial (counts)'][0].vocabulary_))

Multinomial (counts)   acc=0.9620 prec=0.9562 rec=0.9208 f1=0.9382 spec=0.9807 auc=0.9898 cv=0.9701±0.0028  [TP=349 FN=30 FP=16 TN=815]


Bernoulli (binary)     acc=0.9479 prec=0.9675 rec=0.8628 f1=0.9121 spec=0.9868 auc=0.9926 cv=0.9469±0.0072  [TP=327 FN=52 FP=11 TN=820]


Multinomial (TF-IDF)   acc=0.9752 prec=0.9755 rec=0.9446 f1=0.9598 spec=0.9892 auc=0.9975 cv=0.9740±0.0067  [TP=358 FN=21 FP=9 TN=822]
Vocabulary size: 23164


## 6. Most informative words
Each word votes with weight log P(w|spam) − log P(w|ham).

In [6]:
vec, nb = fitted['Multinomial (counts)']
llr = nb.feature_log_prob_[1] - nb.feature_log_prob_[0]
names = vec.get_feature_names_out()
print('Top spam words:', list(names[np.argsort(-llr)[:15]]))
print('Top ham words: ', list(names[np.argsort(llr)[:15]]))

Top spam words: ['enenkio', 'dagga', 'ragga', 'br', 'bgcolor', 'hermios', 'mortgage', 'mso', 'herba', 'kio', 'capillaris', 'eneen', 'annuity', 'botanical', 'atoll']
Top ham words:  ['razor', 'exmh', 'kernel', 'redhat', 'rpm', 'matthias', 'folder', 'perl', 'ximian', 'blog', 'cvs', 'apt', 'weblog', 'unison', 'alsa']


## 7. Decision-threshold trade-off
Raising the threshold reduces false positives (lost real email) at the cost of recall.

In [7]:
prob = fitted['Multinomial (counts)'].predict_proba(Xte)[:, 1]
for th in [0.5, 0.9, 0.99]:
    tn, fp, fn, tp = confusion_matrix(yte, (prob >= th).astype(int)).ravel()
    print(f'threshold={th:<5} FP={fp:<3} FN={fn:<3} recall={tp / (tp + fn):.3f}')

threshold=0.5   FP=16  FN=30  recall=0.921
threshold=0.9   FP=15  FN=33  recall=0.913
threshold=0.99  FP=11  FN=38  recall=0.900
